# Pre-body

## Clearing past runs (optional)

In [ ]:
!rm -rf logs/ # clear logs
!rm -rf spice_out/

## IIC-OSIC Env Setup

In [2]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


## Library Imports

In [3]:
import logging

from pathlib import Path

from symxplorer.spice_engine                import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools              import Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, Project_Setup
from symxplorer.logging                     import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

2025-10-02 06:54:26,221 - SymXplorer.optimizer - Using device: cpu and dtype: torch.float64
2025-10-02 06:54:26,234 - SymXplorer.jupyter - Spicelib_Wrapper imported successfully.


# Instantiations


## Loading the project config

In [4]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

06:54:26 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
06:54:26 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-10-02_06-54-26.log
06:54:26 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
06:54:26 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
06:54:26 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=250, random_seed=48
06:54:26 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
06:54:26 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
06:54:26 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
06:54:26 - SymXplorer.domains: [INFO] 	Number of target specs: 4
06:54:26 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=100e6, range=1.00e+08 tolerance=100000

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(0.01), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(0.01), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(0.001), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(0.001), 'min_res_l': np.float64(1e-06), 'max_ind_size': np.float64(1e-08), 'min

## Create a SPICE simulator wrapper

In [5]:
# (2) Create the Spice Simulator Wrapper
netlist_filename = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.netlist)
output_folder    = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.outdir)

wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename=netlist_filename,
    output_folder=output_folder,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

06:54:26 - SymXplorer.spicelib: [INFO] 📂 Creating output directory for the first time: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
06:54:26 - SymXplorer.spicelib: [INFO] --------------------------------------------------
06:54:26 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
06:54:26 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
06:54:26 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
06:54:26 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
06:54:26 - SymXplorer.spicelib: [INFO] --------------------------------------------------
06:54:26 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
06:54:26 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
06:54:26 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
06:54:26 - SymXplorer.spicelib: [INFO] Testbe

## Create an optimizer object

In [6]:
circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

06:54:26 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 4 target specs


## Sanity Check

In [7]:
# wrapper.run_sanity_check(
#     use_editor=True,
#     sim_execution_t=Sim_Execution_Type.RUN_NOW
# )

# Main Body

## Optimization

In [8]:
circuit_optimizer.parameterize()

Dict(vbias=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_ind_size=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 50.0, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0, 'x_dut_ind_size': 50.0, 'vbias': 50.0}

In [9]:
circuit_optimizer.optimize()

06:54:26 - SymXplorer.optimizer: [INFO] Optimization process started.
06:54:26 - SymXplorer.optimizer: [INFO] Optimizer is set to CMA with budget = 250
Optimizing:   0%|          | 0/250 [00:00<?, ?trial/s]2025-10-02 06:54:26,977 - nevergrad.optimization.optimizerlib - CMA selected CMAbounded optimizer.
06:54:28 - SymXplorer.spicelib: [ERROR] ❌ Variable pm not found in the raw file
Optimizing: 100%|██████████| 250/250 [02:19<00:00,  1.79trial/s]
06:56:46 - SymXplorer.optimizer: [INFO] Optimization process completed.


[{'params': {'x_dut_nfet_w': 63.132471874161396,
   'x_dut_nfet_l': 46.491880471086425,
   'x_dut_cap_w': 38.38927191686664,
   'x_dut_cap_l': 40.86672171893882,
   'x_dut_res_s_l': 53.92597351752503,
   'x_dut_res_s_w': 56.15582335551525,
   'x_dut_res_3_l': 29.5948597042384,
   'x_dut_res_3_w': 47.23087255359133,
   'x_dut_ind_size': 53.057742245200586,
   'vbias': 55.407242154585894},
  'loss': np.float64(1000000000005.799),
  'metadata': {'fc': {'curr_val': np.float64(4316206.0),
    'loss': np.float64(4.039991726209513)},
   'q': {'curr_val': np.float64(27.157564241310748),
    'loss': np.float64(15.157564241310748)},
   'gain_db': {'curr_val': np.float64(-0.548982494550954),
    'loss': np.float64(1.7589642217565737)},
   'pm': {'curr_val': np.float64(nan), 'loss': np.float64(1000000000000.0)}}},
 {'params': {'x_dut_nfet_w': 46.15549538181623,
   'x_dut_nfet_l': 53.19368997087725,
   'x_dut_cap_w': 46.99048553842095,
   'x_dut_cap_l': 49.55777315074815,
   'x_dut_res_s_l': 71.285

In [10]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

06:56:47 - SymXplorer.plotter: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/loss_curve.html
06:56:47 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


## Inspection & Visualization

### (1) Best Param

In [11]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
metadata

06:56:48 - SymXplorer.optimizer: [INFO] best loss: 3.057029532285218


{'fc': {'curr_val': np.float64(89148920.0),
  'loss': np.float64(0.04255374313982818)},
 'q': {'curr_val': np.float64(98.70122450787184),
  'loss': np.float64(86.70122450787184)},
 'gain_db': {'curr_val': np.float64(39.557408354426194),
  'loss': np.float64(0.0)},
 'pm': {'curr_val': np.float64(108.0), 'loss': np.float64(3.01447578914539)}}

In [28]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param] :0.2e}")

x_dut_nfet_w: 8.46e-06
x_dut_nfet_l: 2.31e-06
x_dut_cap_w: 2.15e-03
x_dut_cap_l: 2.25e-05
x_dut_res_s_l: 4.79e-04
x_dut_res_s_w: 2.05e-04
x_dut_res_3_l: 6.94e-04
x_dut_res_3_w: 4.73e-04
x_dut_ind_size: 3.78e-09
vbias: 1.34e+00


In [13]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="vout")

06:56:49 - SymXplorer.optimizer: [INFO] total loss: 3.057029532285218
06:56:49 - SymXplorer.optimizer: [INFO] 	Spec 'fc': curr_val=89148920.0, loss=0.04255374313982818
06:56:49 - SymXplorer.optimizer: [INFO] 	Spec 'q': curr_val=98.70122450787184, loss=86.70122450787184
06:56:49 - SymXplorer.optimizer: [INFO] 	Spec 'gain_db': curr_val=39.557408354426194, loss=0.0
06:56:49 - SymXplorer.optimizer: [INFO] 	Spec 'pm': curr_val=108.0, loss=3.01447578914539


### (3) Metric Trace

In [25]:
circuit_optimizer.plot_optimization_trace(metric_x='pm', metric_y='gain_db', show=True)

06:57:50 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


(tensor([ nan,  nan,  nan,  nan,  nan,  nan,  nan,  nan,  nan,  nan,  nan, 141.,
         128., 138., 144.,   2., 149.,  nan, 144., 137., 138., 123., 143.,  nan,
         145., 139., 130., 160., 113., 147., 123., 135.,  nan, 114., 137., 116.,
         117., 107., 109., 108., 109., 115., 105., 102., 110., 118., 105., 112.,
         105., 110., 107., 111., 105., 107., 103., 125., 110., 104., 103., 100.,
          99.,  99., 106., 109.,  99., 104., 106., 102.,  99., 102., 101., 101.,
         104.,  99., 101., 107., 102., 107., 116., 102., 108., 101., 106., 102.,
         118.,  98., 101., 101., 102., 101., 103., 100., 101., 103.,  99.,  99.,
         103., 101., 101., 100., 106., 103., 106., 104., 116., 100., 104., 109.,
         100.,  99., 101., 103., 100., 100., 101., 108.,  98., 106.,  99.,  99.,
         112., 100., 106., 102.,  99.,  99., 102., 102.,  98., 100., 100., 100.,
         103., 107., 101., 103., 100., 100., 103.,  99.,  99., 100., 103.,  98.,
          97., 102., 100., 1

In [15]:
circuit_optimizer.plot_loss_value_by_spec(spec_name="gain_db", show=True)
circuit_optimizer.plot_loss_value_by_spec(spec_name="fc", show=True)

06:56:49 - SymXplorer.plotter: [INFO] 	min loss 0.0; max loss 2.1642159725996324
06:56:49 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


06:56:49 - SymXplorer.plotter: [INFO] 	min loss 0.0; max loss 4.2564747701679195
06:56:50 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


### (4) Design Space Exploration

In [26]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_w", param_y="x_dut_nfet_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_cap_l", param_y="x_dut_cap_w", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="vbias", param_y="x_dut_ind_size", show=True)

07:00:02 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


07:00:02 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


07:00:02 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


(tensor([0.9973, 0.9313, 0.9922, 1.0248, 0.7825, 0.6337, 1.2381, 0.9699, 0.9048,
         0.8699, 0.8952, 1.2740, 1.4107, 1.0642, 1.0652, 0.8215, 1.3781, 1.3067,
         1.3088, 0.9215, 1.3521, 1.3289, 1.2501, 1.2788, 1.1769, 1.0891, 1.0324,
         1.1793, 1.4549, 1.1649, 1.2565, 1.3306, 1.7319, 1.3508, 1.5201, 1.6036,
         1.6066, 1.4890, 1.4080, 1.4857, 1.3333, 1.5771, 1.5501, 1.3651, 1.5059,
         1.6856, 1.5482, 1.4507, 1.3968, 1.6224, 1.4243, 1.1496, 1.5941, 1.2976,
         1.1598, 1.0578, 0.9300, 1.1200, 1.0947, 1.3881, 1.2690, 1.5350, 0.7913,
         1.3022, 1.3797, 1.0829, 0.9503, 1.2752, 1.5813, 1.1798, 1.0926, 1.3464,
         1.3293, 1.3741, 1.2103, 1.1124, 1.4561, 0.9925, 1.2028, 1.3175, 1.1477,
         1.1078, 1.3683, 1.1245, 0.9109, 1.1196, 1.2600, 1.2368, 1.0730, 1.0208,
         0.9324, 1.0248, 0.9832, 0.9115, 1.0084, 1.0895, 1.0360, 0.9810, 1.0422,
         1.2096, 1.0404, 1.0721, 1.1940, 0.8741, 1.0470, 0.9502, 0.9538, 0.9676,
         1.2317, 1.2178, 1.1

# Testing

## Loss Function Testing

In [17]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

06:56:50 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
06:56:50 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-10-02_06-56-50.log
06:56:50 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
06:56:50 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
06:56:50 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=250, random_seed=48
06:56:50 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
06:56:50 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
06:56:50 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
06:56:50 - SymXplorer.domains: [INFO] 	Number of target specs: 4
06:56:50 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=100e6, range=1.00e+08 tolerance=100000

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(0.01), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(0.01), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(0.001), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(0.001), 'min_res_l': np.float64(1e-06), 'max_ind_size': np.float64(1e-08), 'min

In [18]:
dummy_circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)

import numpy as np

gain_db_min  = np.float64(-135.0)
gain_db_max  = np.float64(135)
points_per_unit = 10
gain_db_vals = np.linspace(gain_db_min, gain_db_max, int((gain_db_max-gain_db_min) * (points_per_unit)))  

for gain_db in gain_db_vals:
    loss, fit_summary = dummy_circuit_optimizer.compute_fitness(performance_array={'gain_db' : gain_db})
    dummy_circuit_optimizer.optimization_log.append({
            "metric_value": None,
            "fit_summary": fit_summary,
            "params": None,
            "log": None
        })

dummy_circuit_optimizer.plot_loss_value_by_spec(spec_name='gain_db', show = True)

06:56:50 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 4 target specs
06:56:52 - SymXplorer.plotter: [INFO] 	min loss 0.0; max loss 6.910694698329305
06:56:52 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


In [19]:
dummy_circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)

import numpy as np

fc_min  = 4
fc_max  = 10
points_per_unit = 1000

# num_points = int((fc_max-fc_min) * (points_per_unit/1e3))
logger.info(f"using {points_per_unit} points")
logger.info(f"\tTarget: {PROJECT_SETUP.optimizer_config.target_specs.get_target_by_name("fc")}")
# fc_vals = np.logspace(fc_min, fc_max)  
fc_vals = np.linspace(1e1, 1e8, 1000)  

for fc in fc_vals:
    loss, fit_summary = dummy_circuit_optimizer.compute_fitness(performance_array={'fc' : fc})
    dummy_circuit_optimizer.optimization_log.append({
            "metric_value": None,
            "fit_summary": fit_summary,
            "params": None,
            "log": None
        })

dummy_circuit_optimizer.plot_loss_value_by_spec(spec_name='fc', show = True)

06:56:52 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 4 target specs
06:56:52 - SymXplorer.jupyter: [INFO] using 1000 points
06:56:52 - SymXplorer.jupyter: [INFO] 	Target: TargetSpec(name=fc, target=100e6, range=1.00e+08 tolerance=10000000.0, goal=exact, sim_type=ac, enable=True, error_type=relative-sigmoid, weight=10.0, enable=True, description=Center frequency)
06:56:53 - SymXplorer.plotter: [INFO] 	min loss 0.0; max loss 4.218989641499455
06:56:53 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


## Other

In [20]:
circuit_optimizer.optimization_log

[{'metric_value': np.float64(1000000000005.799),
  'fit_summary': {'fc': {'curr_val': np.float64(4316206.0),
    'loss': np.float64(4.039991726209513)},
   'q': {'curr_val': np.float64(27.157564241310748),
    'loss': np.float64(15.157564241310748)},
   'gain_db': {'curr_val': np.float64(-0.548982494550954),
    'loss': np.float64(1.7589642217565737)},
   'pm': {'curr_val': np.float64(nan), 'loss': np.float64(1000000000000.0)}},
  'params': {'x_dut_nfet_w': 6.379608738042649e-06,
   'x_dut_nfet_l': 4.7455026622606865e-06,
   'x_dut_cap_w': 0.003839543298967496,
   'x_dut_cap_l': 0.0040872635046766935,
   'x_dut_res_s_l': 0.0005397204754400752,
   'x_dut_res_s_w': 0.0005619966753215975,
   'x_dut_res_3_l': 0.00029665264844534164,
   'x_dut_res_3_w': 0.00047283641681037747,
   'x_dut_ind_size': 5.775196802068053e-09,
   'vbias': 0.9973303587825461},
  'log': PosixPath('/foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/run_1/tb_ac_1.log')},
 {'metric_value': np.fl

In [21]:
PROJECT_SETUP.optimizer_config.target_specs.list_target_names()

['fc', 'q', 'gain_db', 'pm']

In [22]:
target_spec = PROJECT_SETUP.optimizer_config.target_specs.get_target_by_name('gain_db')
target_spec

TargetSpec(name='gain_db', target=40, goal=<OptimizationGoalType.EXCEED: 'exceed'>, sim_type=<SimType.AC: 'ac'>, log_scale=False, enable=True, range=np.float64(100.0), error_type=<Error_Types.RELATIVE_SIGMOID: 'relative-sigmoid'>, weight=10.0, tolerance=5, description='gain in dB at fc')

In [23]:
circuit_optimizer.compute_spec_loss(curr_val=-90, target_spec=target_spec)

np.float64(5.5459972234938215)

In [24]:
PROJECT_SETUP.dut_params

[Param(name='x_dut_nfet_w', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False),
 Param(name='x_dut_nfet_l', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False),
 Param(name='x_dut_cap_w', min_val=np.float64(1e-06), max_val=np.float64(0.01), val=None, description=None, log_scale=False),
 Param(name='x_dut_cap_l', min_val=np.float64(1e-06), max_val=np.float64(0.01), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_s_l', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_s_w', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_3_l', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_3_w', min_val=np.float64(1e-06), max_val=np.fl